## Introduction

The purpose of this notebook is to demonstrate the usage of the `vib_frags` package for obtaining the (BLISS-)Pauli LCU of two-mode coupling vibrational Hamiltonians, as well as calculating the associated costs for QPE.

## Importing modules

In [ ]:
import numpy as np
import h5py

from vib_frags import bliss_pauli as bp
from vib_frags import bliss_utils as bu
from vib_frags import tensor_utils as tu
from vib_frags import cost_utils as cu
%load_ext autoreload
%autoreload 2

## Importing Hamiltonian

Hamiltonians used for Trotter estimates are smaller and are stored in a different convention from the PAH Hamiltonians. That distinction is made in how the loaded tensors are transposed.

In [ ]:
mol_name = 'H2S'                                                     #Can be 'pah1', 'pah2', ..., 'pah7', or 'H2S', 'CO2', 'CH2O', 'CH4' 
au_to_cm1 = 219474.6                                                  #Stored tensors are in atomic units
mc_level = 2                                                          #Mode-coupling level
nmodals = 4                                                           #Number of modals
Ham_source = 'Trotter'                                                    #Can be 'Trotter' or 'PAH'. In the former case, the Hamiltonians considered are much smaller than the PAH Hamiltonians

#Loading the Hamiltonian
if Ham_source == 'Trotter':
  datafile = h5py.File(rf'Hamiltonians/Trotter_Hamiltonians/{mol_name}_loc.hdf5', 'r')
elif Ham_source == 'PAH':
  datafile = h5py.File(rf'Hamiltonians/PAH_Hamiltonians/{mol_name}/{mol_name}_{mc_level}mode/{mol_name}_VSCF_{nmodals}M_{mc_level}MC.hdf5', 'r')

#Extracting Hamiltonian terms at different mode-coupling levels
#One-mode-coupling term
H1_full = np.array(datafile['H1'][()]) * au_to_cm1     
H1 = H1_full[:, :nmodals, :nmodals]

#Two-mode-coupling term
H2_full = np.array(datafile['H2'][()]) * au_to_cm1      
if Ham_source == 'Trotter':
  H2_nonsym = np.transpose(H2_full, (0,2,4,1,3,5))                          #Permuting axes to match the convention in the paper
elif Ham_source == 'PAH':
  H2_nonsym = np.transpose(H2_full, (3,4,5,0,1,2))
H2_nonsym = H2_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals]       #Non symmetrized tensor
H2 = tu.symmetrize_tbt(H2_nonsym)

#Three-mode-coupling term
if Ham_source == 'PAH':
  try:
    H3_full = np.array(datafile['H3'][()]) * au_to_cm1
    H3_nonsym = np.transpose(H3_full, (6,7,8,3,4,5,0,1,2))
    H3_nonsym = H3_nonsym[:, :nmodals, :nmodals, :, :nmodals, :nmodals, :, :nmodals, :nmodals]
    H3 = tu.symmetrize_trbt(H3_nonsym)
  except KeyError:
    print ("3 mode term is missing. Setting 3 mode tensor to None.")
    H3_full, H3_nonsym, H3 = None, None, None
elif Ham_source == 'Trotter':
  H3_full, H3_nonsym, H3 = None, None, None
  mc_level = 2

nmodes = H1.shape[0]
print ('Number of modes:', nmodes)
print ('Number of modals:', nmodals)

## BLISS-Pauli

By setting syms=[] provides the 1-norm for the ordinary Pauli LCU. The symmetries A, B , C, etc can be understood from associated appendix of the paper.

In [ ]:
H0_opt, H1_opt, H2_opt, H3_opt, bparams_dict = bp.optimize_bliss_pauli(obt=H1, tbt=H2, trbt=None, learning_rate=1e-1, syms=['A', 'B', 'C'], initial_guess=None, maxiter=50000, chunk_size=500, ret_params=True)

## Pauli QPE cost

In [ ]:
#cutoff to prune LCU
MC = mc_level
bliss = True

if bliss:
  if MC == 2:
    cutoff_dict = {'pah1': 0.012, 'pah2': 0.00894, 'pah3': 0.00643, 'pah4': 0.00574, 'pah5': 0.00477, 'pah6': 0.0039, 'pah7': 0.00342, 'H2S': 11, 'CO2': 16, 'CH2O': 18, 'CH4': 10}
  else:
    cutoff_dict = {'pah1': 0.00313, 'pah2': 0.001682}
else:
  if MC == 2:
    cutoff_dict = {'pah1': 0.0113, 'pah2': 0.00885, 'pah3': 0.00588, 'pah4': 0.00543, 'pah5': 0.00438, 'pah6': 0.00339, 'pah7': 0.0029}
  else:
    cutoff_dict = {'pah1': 0.003233, 'pah2': 0.00192}
cutoff = cutoff_dict[mol_name]

if bliss == True:
  H0B, H1_main, H2_main, H3_main = bu.get_bliss_hamiltonian(H1, tbt=H2, trbt=None, params=bparams_dict)
  H2_nonsym_main = tu.unsymmetrize_tbt(H2_main)
  H3_nonsym_main = None
else:
  H1_main, H2_main, H3_main = H1, H2, None
  H2_nonsym_main, H3_nonsym_main = H2_nonsym, None

#Number of unqiue LCU terms
S = cu.get_Pauli_S(H1_main, H2_nonsym_main, H3_nonsym_main, cutoff)

#Pruning small coefficients
H1_main_approx, H2_nonsym_main_approx, H3_nonsym_main_approx, sparse_ten_error = cu.get_pruned_tensors_w_err(H1_main, H2_nonsym_main, H3_nonsym_main, mc_level, cutoff)
H2_main_approx = tu.symmetrize_tbt(H2_nonsym_main_approx) 
H3_main_approx = tu.symmetrize_trbt(H3_nonsym_main_approx)

print ("cutoff used = ", cutoff)
eps_ten = np.round(sparse_ten_error, 2)
rel_error = np.linalg.norm(H2_nonsym_main_approx - H2_nonsym_main)/np.linalg.norm(H2_nonsym_main)

one_norm = tu.opt_Pauli_LCU_1norm(obt=H1_main, tbt=H2_main_approx, trbt=H3_main_approx)
eps = 10

print ("Number of LCU coefficients to load: ", S)

QROAM_type = "dirty"
select_copy = False

if QROAM_type == "dirty":
  dirty, clean = True, False
elif QROAM_type == "clean":
  dirty, clean = False, 605
elif (QROAM_type == "None") or (QROAM_type == "QROM") or (QROAM_type == "False"):
  dirty, clean = False, False
else:
  raise("Invalid QROAM type provided")

C_Pauli, N_Pauli = cu.get_Pauli_QPE_cost(one_norm, eps, nmodes, nmodals, S, mc_level, br = 20, aleph = 20, dirty = dirty, clean = clean, select_copy = select_copy)

print ("\nMolecule: ", mol_name)
print ("Modals per mode: ", nmodals)
print ("\nRelative error in tensor approximation:", np.round(rel_error,7), "cm-1")
print (f"Absolute error in tensor approximation: {eps_ten} cm-1")
print (f"Target accuracy in energy: {eps} cm-1")